# MIND-small: dataset EDA

Retains the data quality, funnel, sparsity and temporal analysis from Dataset_Loading_Parsing. Preparation for modelling lives in `../01_Data_Preprocessing.ipynb`. Click totals below are diagnostics, never model inputs.

In [ ]:
from pathlib import Path
import os
import sys

# In Colab, clone/open the repository first; set BT4222_PROJECT_ROOT if needed.
# For shared Drive data, set MIND_RAW_DIR to the folder containing MINDsmall_train.
try:
    from google.colab import drive
except ImportError:
    pass
else:
    drive.mount("/content/drive")

configured_root = os.environ.get("BT4222_PROJECT_ROOT")
if configured_root:
    PROJECT_ROOT = Path(configured_root).expanduser().resolve()
else:
    PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                         if (p / "src/data_loading.py").is_file()), None)
if PROJECT_ROOT is None or not (PROJECT_ROOT / "src/data_loading.py").is_file():
    raise FileNotFoundError("Open the cloned repo, or set BT4222_PROJECT_ROOT to its path.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DATA_ROOT = Path(os.environ.get("MIND_RAW_DIR", str(PROJECT_ROOT / "data/raw"))).expanduser()

import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.preprocessing import (
    BEHAVIORS_COLUMNS, NEWS_COLUMNS, read_sampling_tsv,
    prepare_events, clean_news,
)

DATA_DIR = DATA_ROOT / "MINDsmall_train"
behaviour_cols = BEHAVIORS_COLUMNS
news_cols = NEWS_COLUMNS
behaviours = prepare_events(read_sampling_tsv(DATA_DIR / "behaviors.tsv", behaviour_cols),
                            "MINDsmall_train/behaviors.tsv")
# Aliases preserve report cells without a second parsing implementation.
behaviours["time_parsed"] = behaviours["timestamp"]
behaviours["history_list"] = behaviours["history_ids"]
behaviours["parsed_impressions"] = behaviours["impressions"].map(
    lambda value: [{"news_id": token.rsplit("-", 1)[0], "label": int(token.rsplit("-", 1)[1])}
                   for token in value.split()])
behaviours["num_candidates"] = behaviours["parsed_impressions"].str.len()
behaviours["num_clicked"] = behaviours["parsed_impressions"].map(lambda entries: sum(e["label"] for e in entries))
behaviours["num_unclicked"] = behaviours["num_candidates"] - behaviours["num_clicked"]
# Keep raw missing values for EDA summaries; copy derived columns from cleaning.
news = read_sampling_tsv(DATA_DIR / "news.tsv", news_cols)
prepared_news = clean_news(news)
for column in prepared_news.columns.difference(news.columns):
    news[column] = prepared_news[column]
def load_embedding(path, id_col):
    df = pd.read_csv(
        path,
        sep="\t",
        header=None
    )

    # Remove completely empty columns, e.g. caused by trailing tabs
    df = df.dropna(axis=1, how="all")

    embedding_dim = df.shape[1] - 1

    df.columns = (
        [id_col]
        + [f"emb_{i}" for i in range(embedding_dim)]
    )

    # Ensure embeddings are numeric
    embedding_cols = [c for c in df.columns if c != id_col]
    df[embedding_cols] = df[embedding_cols].apply(
        pd.to_numeric,
        errors="coerce"
    )

    return df
entity_embeddings = load_embedding(DATA_DIR / "entity_embedding.vec", "entity_id")
relation_embeddings = load_embedding(DATA_DIR / "relation_embedding.vec", "relation_id")
print("Behaviours:", behaviours.shape, "News:", news.shape)


## Data Quality and Missing Values

In [ ]:
def missing_summary(df):
    summary = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": df.isna().mean() * 100
    })

    return summary.sort_values(
        "missing_pct",
        ascending=False
    )

In [ ]:
print("Behaviors missing values")
display(missing_summary(
    behaviours[behaviour_cols]
))

print("News missing values")
display(missing_summary(
    news[news_cols]
))

### Duplicates

In [ ]:
print("Duplicate behavior rows:",
      behaviours.duplicated(subset=behaviour_cols).sum())

print("Duplicate impression IDs:",
      behaviours["impression_id"].duplicated().sum())

print("Duplicate news rows:",
      news.duplicated(subset=news_cols).sum())

print("Duplicate news IDs:",
      news["news_id"].duplicated().sum())

print("Duplicate entity IDs:",
      entity_embeddings["entity_id"].duplicated().sum())

print("Duplicate relation IDs:",
      relation_embeddings["relation_id"].duplicated().sum())

### Identifier validity

In [ ]:
invalid_users = (
    ~behaviours["user_id"]
    .fillna("")
    .str.match(r"^U\d+$")
)

invalid_news = (
    ~news["news_id"]
    .fillna("")
    .str.match(r"^N\d+$")
)

print("Invalid user IDs:", invalid_users.sum())
print("Invalid news IDs:", invalid_news.sum())

### Timestamp validity

In [ ]:
print(
    "Unparseable timestamps:",
    behaviours["time_parsed"].isna().sum()
)

print(
    "Timestamp range:",
    behaviours["time_parsed"].min(),
    "to",
    behaviours["time_parsed"].max()
)

### Impression-format validation

In [ ]:
impression_pattern = re.compile(r"^N\d+-[01]$")

invalid_impression_tokens = []

for impression_string in behaviours["impressions"].dropna():

    for token in impression_string.split():

        if not impression_pattern.match(token):
            invalid_impression_tokens.append(token)

print(
    "Invalid impression tokens:",
    len(invalid_impression_tokens)
)

### Click-class imbalance

In [ ]:
total_clicked = behaviours["num_clicked"].sum()
total_unclicked = behaviours["num_unclicked"].sum()

total_candidates = (
    total_clicked + total_unclicked
)

click_rate = (
    total_clicked / total_candidates
)

print("Clicked candidates:", total_clicked)
print("Unclicked candidates:", total_unclicked)
print("Total candidates:", total_candidates)
print(f"Click rate: {click_rate:.2%}")

In [ ]:
import matplotlib.pyplot as plt

clicked = behaviours["num_clicked"].sum()
unclicked = behaviours["num_unclicked"].sum()

labels = ["Clicked", "Unclicked"]
counts = [clicked, unclicked]

fig, ax = plt.subplots(figsize=(7, 5))

bars = ax.bar(labels, counts)

ax.set_title("Distribution of Clicked and Unclicked Candidate Impressions")
ax.set_ylabel("Number of Candidate Impressions")

# Add counts and percentages above bars
total = sum(counts)

for bar, count in zip(bars, counts):
    percentage = count / total * 100

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{count:,}\n({percentage:.2f}%)",
        ha="center",
        va="bottom"
    )

# Makes large numbers easier to read
ax.ticklabel_format(
    style="plain",
    axis="y"
)

plt.tight_layout()
plt.show()

### History & candidate distribution analysis

In [ ]:
display(
    behaviours[
        [
            "history_length",
            "num_candidates",
            "num_clicked"
        ]
    ].describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

In [ ]:
history_99 = behaviours["history_length"].quantile(0.99)
history_median = behaviours["history_length"].median()

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    behaviours.loc[
        behaviours["history_length"] <= history_99,
        "history_length"
    ],
    bins=40
)

ax.axvline(
    history_median,
    linestyle="--",
    linewidth=1.5,
    label=f"Median = {history_median:.0f}"
)

ax.set_title("Distribution of User History Lengths")
ax.set_xlabel("Number of Previously Clicked Articles")
ax.set_ylabel("Number of Impressions")
ax.legend()

plt.tight_layout()
plt.show()

### News category distributions

In [ ]:
print("Number of categories:",
      news["category"].nunique())

print("Number of subcategories:",
      news["subcategory"].nunique())

display(
    news["category"]
    .value_counts()
    .to_frame("count")
)

display(
    news["subcategory"]
    .value_counts()
    .head(20)
    .to_frame("count")
)

In [ ]:
category_counts = (
    news["category"]
    .value_counts()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(8, 6))

bars = ax.barh(
    category_counts.index,
    category_counts.values
)

ax.set_title("Distribution of News Articles by Category")
ax.set_xlabel("Number of Articles")
ax.set_ylabel("Category")

# Add count labels
for bar in bars:
    width = bar.get_width()

    ax.text(
        width,
        bar.get_y() + bar.get_height() / 2,
        f" {int(width):,}",
        va="center"
    )

plt.tight_layout()
plt.show()

### News text quality

In [ ]:
display(
    news[
        [
            "title_length",
            "abstract_length",
            "num_title_entities",
            "num_abstract_entities"
        ]
    ].describe()
)

In [ ]:
print(
    "Empty/missing titles:",
    news["title"].fillna("").str.strip().eq("").sum()
)

print(
    "Empty/missing abstracts:",
    news["abstract"].fillna("").str.strip().eq("").sum()
)

### Check malformed entity JSON

In [ ]:
invalid_title_json = (
    news["title_entities_parsed"]
    .apply(lambda x: x is None)
    .sum()
)

invalid_abstract_json = (
    news["abstract_entities_parsed"]
    .apply(lambda x: x is None)
    .sum()
)

print(
    "Malformed title entity JSON:",
    invalid_title_json
)

print(
    "Malformed abstract entity JSON:",
    invalid_abstract_json
)

### Referential integrity betewen behaviours and news

In [ ]:
behaviour_news_ids = set()

# IDs appearing in user histories
for history in behaviours["history"].dropna():
    behaviour_news_ids.update(history.split())

# IDs appearing in candidate impressions
for impressions in behaviours["impressions"].dropna():

    for token in impressions.split():
        news_id = token.rsplit("-", 1)[0]
        behaviour_news_ids.add(news_id)

news_ids = set(news["news_id"])

missing_from_news = (
    behaviour_news_ids - news_ids
)

print(
    "Unique news IDs referenced in behaviours:",
    len(behaviour_news_ids)
)

print(
    "Referenced IDs absent from news.tsv:",
    len(missing_from_news)
)

### Check entity-embedding coverage

In [ ]:
def extract_wikidata_ids(entity_list):

    if not isinstance(entity_list, list):
        return []

    return [
        entity["WikidataId"]
        for entity in entity_list
        if "WikidataId" in entity
    ]

In [ ]:
news["title_wikidata_ids"] = (
    news["title_entities_parsed"]
    .apply(extract_wikidata_ids)
)

news["abstract_wikidata_ids"] = (
    news["abstract_entities_parsed"]
    .apply(extract_wikidata_ids)
)

In [ ]:
news_entity_ids = set()

for ids in news["title_wikidata_ids"]:
    news_entity_ids.update(ids)

for ids in news["abstract_wikidata_ids"]:
    news_entity_ids.update(ids)

embedding_entity_ids = set(
    entity_embeddings["entity_id"]
)

entities_without_embedding = (
    news_entity_ids - embedding_entity_ids
)

print(
    "Unique entities referenced in news:",
    len(news_entity_ids)
)

print(
    "Entities with embeddings:",
    len(news_entity_ids & embedding_entity_ids)
)

print(
    "Entities without embeddings:",
    len(entities_without_embedding)
)

if len(news_entity_ids) > 0:
    print(
        "Entity embedding coverage:",
        f"{len(news_entity_ids & embedding_entity_ids) / len(news_entity_ids):.2%}"
    )

### Embedding quality

In [ ]:
def check_embedding_quality(df, id_col):

    embedding_cols = [
        col for col in df.columns
        if col != id_col
    ]

    values = df[embedding_cols].to_numpy()

    return pd.Series({
        "rows": len(df),
        "dimensions": len(embedding_cols),
        "duplicate_ids": df[id_col].duplicated().sum(),
        "missing_values":
            df[embedding_cols].isna().sum().sum(),
        "non_finite_values":
            (~np.isfinite(values)).sum()
    })

In [ ]:
embedding_quality = pd.DataFrame({
    "entity_embedding": check_embedding_quality(
        entity_embeddings,
        "entity_id"
    ),
    "relation_embedding": check_embedding_quality(
        relation_embeddings,
        "relation_id"
    )
}).T

display(embedding_quality)

## Summary

In [ ]:
dataset_summary = pd.DataFrame({
    "Dataset": [
        "Behaviors",
        "News",
        "Entity embeddings",
        "Relation embeddings"
    ],
    "Rows": [
        len(behaviours),
        len(news),
        len(entity_embeddings),
        len(relation_embeddings)
    ],
    "Columns": [
        len(behaviour_cols),
        len(news_cols),
        entity_embeddings.shape[1],
        relation_embeddings.shape[1]
    ]
})

display(dataset_summary)

In [ ]:
feature_metadata = pd.DataFrame([
    {
        "Feature": "impression_id",
        "Source": "behaviors.tsv",
        "Description": "Identifier for an impression/recommendation event",
        "Data type / unit": "Identifier",
        "Status": "Raw"
    },
    {
        "Feature": "user_id",
        "Source": "behaviors.tsv",
        "Description": "Anonymised user identifier",
        "Data type / unit": "Categorical identifier",
        "Status": "Raw"
    },
    {
        "Feature": "time",
        "Source": "behaviors.tsv",
        "Description": "Timestamp of the impression event",
        "Data type / unit": "Datetime",
        "Status": "Raw"
    },
    {
        "Feature": "history",
        "Source": "behaviors.tsv",
        "Description": "Previously clicked news articles",
        "Data type / unit": "Sequence of news IDs",
        "Status": "Raw"
    },
    {
        "Feature": "category",
        "Source": "news.tsv",
        "Description": "Top-level news category",
        "Data type / unit": "Categorical",
        "Status": "Raw"
    },
    {
        "Feature": "subcategory",
        "Source": "news.tsv",
        "Description": "News subcategory",
        "Data type / unit": "Categorical",
        "Status": "Raw"
    },
    {
        "Feature": "title",
        "Source": "news.tsv",
        "Description": "Article headline",
        "Data type / unit": "Text",
        "Status": "Raw"
    },
    {
        "Feature": "abstract",
        "Source": "news.tsv",
        "Description": "Article summary",
        "Data type / unit": "Text",
        "Status": "Raw"
    },
    {
        "Feature": "history_length",
        "Source": "behaviors.tsv",
        "Description": "Number of previously clicked articles",
        "Data type / unit": "Count",
        "Status": "Derived"
    },
    {
        "Feature": "num_candidates",
        "Source": "behaviors.tsv",
        "Description": "Number of candidate articles in an impression",
        "Data type / unit": "Count",
        "Status": "Derived"
    },
    {
        "Feature": "num_clicked",
        "Source": "behaviors.tsv",
        "Description": "Number of clicked candidates in an impression",
        "Data type / unit": "Count",
        "Status": "Derived"
    },
    {
        "Feature": "num_unclicked",
        "Source": "behaviors.tsv",
        "Description": "Number of exposed but unclicked candidates",
        "Data type / unit": "Count",
        "Status": "Derived"
    },
    {
        "Feature": "title_length",
        "Source": "news.tsv",
        "Description": "Number of words in article title",
        "Data type / unit": "Words",
        "Status": "Derived"
    },
    {
        "Feature": "abstract_length",
        "Source": "news.tsv",
        "Description": "Number of words in article abstract",
        "Data type / unit": "Words",
        "Status": "Derived"
    },
    {
        "Feature": "num_title_entities",
        "Source": "news.tsv",
        "Description": "Number of entities identified in title",
        "Data type / unit": "Count",
        "Status": "Derived"
    },
    {
        "Feature": "num_abstract_entities",
        "Source": "news.tsv",
        "Description": "Number of entities identified in abstract",
        "Data type / unit": "Count",
        "Status": "Derived"
    },
    {
        "Feature": "entity_embedding",
        "Source": "entity_embedding.vec",
        "Description": "Pretrained representation of Wikidata entity",
        "Data type / unit": "100-dimensional vector",
        "Status": "Raw / pretrained"
    },
    {
        "Feature": "title_abstract_tfidf",
        "Source": "news.tsv",
        "Description": "TF-IDF representation of article textual content",
        "Data type / unit": "Numeric vector",
        "Status": "Proposed"
    },
    {
        "Feature": "historical_popularity_ctr",
        "Source": "behaviors.tsv",
        "Description": "Article popularity/click-through signal computed using prior interactions only",
        "Data type / unit": "Count / proportion",
        "Status": "Proposed"
    }
])

display(feature_metadata)

In [ ]:
numeric_features = {
    "history_length": behaviours["history_length"],
    "num_candidates": behaviours["num_candidates"],
    "num_clicked": behaviours["num_clicked"],
    "num_unclicked": behaviours["num_unclicked"],
    "title_length": news["title_length"],
    "abstract_length": news["abstract_length"],
    "num_title_entities": news["num_title_entities"],
    "num_abstract_entities": news["num_abstract_entities"]
}

numeric_summary = []

for feature, series in numeric_features.items():
    numeric_summary.append({
        "Feature": feature,
        "Min": series.min(),
        "Max": series.max(),
        "Mean": series.mean(),
        "Median": series.median(),
        "Std. Dev.": series.std(),
        "Missing": series.isna().sum()
    })

numeric_summary = pd.DataFrame(numeric_summary)

# Make table easier to read
numeric_summary[
    ["Mean", "Median", "Std. Dev."]
] = numeric_summary[
    ["Mean", "Median", "Std. Dev."]
].round(2)

display(numeric_summary)

## 4.2 Interaction Hierarchy & Funnel

In [ ]:
# Funnel for MINDsmall_train: exposure -> click


total_impressions = len(behaviours)
total_exposures = behaviours["num_candidates"].sum()
total_clicks = behaviours["num_clicked"].sum()

overall_click_rate = total_clicks / total_exposures

funnel = pd.DataFrame({
    "Stage": ["Exposure (candidate shown)", "Click"],
    "Count": [total_exposures, total_clicks],
    "% of exposures": [100.0, overall_click_rate * 100]
})

print("Total impressions (sessions):", f"{total_impressions:,}")
print("Total candidate exposures:", f"{total_exposures:,}")
print("Total clicks:", f"{total_clicks:,}")
print(f"Overall click rate: {overall_click_rate:.4f} ({overall_click_rate * 100:.2f}%)")

display(funnel)

In [ ]:
# Distribution of clicks per impression
clicks_per_impression = (
    behaviours["num_clicked"]
    .value_counts()
    .sort_index()
    .to_frame("num_impressions")
)

clicks_per_impression["pct_impressions"] = (
    clicks_per_impression["num_impressions"] / total_impressions * 100
)

clicks_per_impression.index.name = "clicks_in_impression"

display(
    behaviours["num_clicked"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

display(clicks_per_impression)

In [ ]:
# Group long tail into a "5+" bucket for readability
click_buckets = (
    behaviours["num_clicked"]
    .clip(upper=5)
    .value_counts()
    .sort_index()
)

bucket_labels = [
    "5+" if k == 5 else str(k)
    for k in click_buckets.index
]

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(bucket_labels, click_buckets.values)

for bar, count in zip(bars, click_buckets.values):
    percentage = count / total_impressions * 100

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.1,
        f"{count:,}\n({percentage:.2f}%)",
        ha="center",
        va="bottom"
    )

ax.set_title("Distribution of Clicks per Impression")
ax.set_xlabel("Number of Clicked Candidates in Impression")
ax.set_ylabel("Number of Impressions")

ax.ticklabel_format(
    style="plain",
    axis="y"
)
plt.show()

## 4.4 Sparsity & Temporal Patterns

Run the existing loading and parsing cells first. These cells use `behaviours` (British spelling) and `news`; they do not modify those dataframes or source files.

Scope: the currently loaded, **unsampled** source split. For this notebook, use the original **MINDsmall_train** (all 9–14 November events), before your group's further chronological split. Keep the reserved final evaluation set separate.

We examine supplied history lengths per impression, article exposure/click concentration, and gaps between each user's logged impressions. These descriptive measures are not a user–article matrix density estimate. Histories are not lifetime activity records, and impression gaps are not intervals between individual clicks.


In [ ]:
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

s44_required = [
    "impression_id", "user_id", "time_parsed", "history_length",
    "parsed_impressions", "num_candidates", "num_clicked",
]
s44_missing = sorted(set(s44_required) - set(behaviours.columns))
if s44_missing:
    raise ValueError(f"Run the earlier parsing cells first. Missing columns: {s44_missing}")
s44 = behaviours[s44_required].copy()
if s44.empty:
    raise ValueError("The loaded behaviours table is empty.")
if s44[["impression_id", "user_id", "time_parsed", "history_length"]].isna().any().any():
    raise ValueError("Resolve missing IDs, invalid timestamps or unparsed history lengths first.")
if s44["impression_id"].duplicated().any():
    raise ValueError("Repeated impression IDs found. Review duplicates before counting events.")
if (s44["history_length"] < 0).any():
    raise ValueError("History lengths cannot be negative.")

s44_scope = Path(DATA_DIR).name if "DATA_DIR" in globals() else "loaded source split"
print(f"Scope: {s44_scope} — use unsampled impressions")
print(f"{len(s44):,} impressions; {s44['user_id'].nunique():,} distinct users")
print(f"Recorded time range: {s44['time_parsed'].min()} to {s44['time_parsed'].max()}")


### 4.4.1 Supplied click-history lengths

Each observation is the history available at one impression. Frequent users contribute more observations: this is **not** a distribution with one observation per user. Empty histories indicate no supplied prior clicks for that event, not necessarily no lifetime activity.

Also summarize logged impressions per user. The histogram includes zero and the full history-length range; its vertical axis uses a logarithmic scale.


In [ ]:
s44_history = s44["history_length"]
s44_impressions_per_user = s44.groupby("user_id").size()
s44_history_summary = pd.DataFrame({
    "History length per impression": s44_history.describe(
        percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]),
    "Logged impressions per user": s44_impressions_per_user.describe(
        percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]),
})
s44_empty_histories = int(s44_history.eq(0).sum())
s44_empty_history_pct = 100 * s44_empty_histories / len(s44)
display(s44_history_summary.round(2))
print(f"Empty supplied histories: {s44_empty_histories:,} / {len(s44):,} impressions "
      f"({s44_empty_history_pct:.2f}%)")

s44_fig_history, ax = plt.subplots(figsize=(8, 4))
ax.hist(s44_history, bins=50, color="#2878A5", edgecolor="white", log=True)
ax.axvline(s44_history.median(), color="#C04D33", linestyle="--",
           label=f"Median = {s44_history.median():g}")
ax.set(title="Supplied click-history lengths",
       xlabel="Number of article IDs in the supplied history",
       ylabel="Number of impressions (log scale)")
ax.legend()
s44_fig_history.tight_layout()
plt.show()


### 4.4.2 Article popularity: head versus long tail

Count candidate entries as exposures and label-1 entries as clicks. Do **not** add history entries to these click counts: histories recur across impressions and describe earlier activity.

Concentration uses all distinct **exposed articles**, including zero-click articles. Metadata-only articles are reported separately; their presence does not establish availability for every impression.

Exposure and click curves rank articles **separately**, so their “top 10%” groups may differ. Round fractions up to a whole article and report the actual count.


In [ ]:
s44_exposure_counts, s44_click_counts = Counter(), Counter()
# Count directly from parsed lists; no large candidate-level dataframe is needed.
for candidates in s44["parsed_impressions"]:
    for candidate in candidates:
        article_id, label = candidate["news_id"], candidate["label"]
        if label not in (0, 1):
            raise ValueError(f"Unexpected click label: {label!r}")
        if pd.isna(article_id) or str(article_id).strip() == "":
            raise ValueError("A candidate is missing its article ID.")
        s44_exposure_counts[article_id] += 1
        if label == 1:
            s44_click_counts[article_id] += 1
if not s44_exposure_counts:
    raise ValueError("No candidate exposures were found.")

s44_total_exposures = sum(s44_exposure_counts.values())
s44_total_clicks = sum(s44_click_counts.values())
assert s44_total_exposures == int(s44["num_candidates"].sum()), "Exposure totals disagree."
assert s44_total_clicks == int(s44["num_clicked"].sum()), "Click totals disagree."

s44_metadata_ids = set(news["news_id"].dropna())
s44_all_ids = sorted(s44_metadata_ids | set(s44_exposure_counts))
s44_article_counts = pd.DataFrame({
    "news_id": s44_all_ids,
    "exposures": [s44_exposure_counts[x] for x in s44_all_ids],
    "clicks": [s44_click_counts[x] for x in s44_all_ids],
    "has_metadata": [x in s44_metadata_ids for x in s44_all_ids],
})
s44_exposed_articles = s44_article_counts.loc[s44_article_counts["exposures"] > 0].copy()
s44_n_articles = len(s44_exposed_articles)
s44_zero_click_articles = int(s44_exposed_articles["clicks"].eq(0).sum())
s44_popularity_summary = pd.Series({
    "Distinct exposed articles": s44_n_articles,
    "Distinct articles with at least one click": int(s44_exposed_articles["clicks"].gt(0).sum()),
    "Exposed articles with zero clicks": s44_zero_click_articles,
    "Zero-click share of exposed articles (%)": 100 * s44_zero_click_articles / s44_n_articles,
    "Metadata articles with no recorded exposure": int(
        ((s44_article_counts["exposures"] == 0) & s44_article_counts["has_metadata"]).sum()),
    "Exposed articles missing from news metadata": int(
        (~s44_exposed_articles["has_metadata"]).sum()),
}, name="Value").to_frame()
display(s44_popularity_summary.round(2))

s44_concentration_rows = []
for fraction in (0.01, 0.10, 0.20):
    k = max(1, int(np.ceil(fraction * s44_n_articles)))
    s44_concentration_rows.append({
        "Top fraction": f"{fraction:.0%}",
        "Number of articles": k,
        "Actual article share (%)": 100 * k / s44_n_articles,
        "Exposure share (ranked by exposures, %)":
            100 * s44_exposed_articles["exposures"].nlargest(k).sum() / s44_total_exposures,
        "Click share (ranked by clicks, %)":
            (100 * s44_exposed_articles["clicks"].nlargest(k).sum() / s44_total_clicks)
            if s44_total_clicks else np.nan,
    })
s44_concentration = pd.DataFrame(s44_concentration_rows).set_index("Top fraction")
display(s44_concentration.round(2))

s44_fig_popularity, ax = plt.subplots(figsize=(7, 4.5))
s44_rank_pct = np.arange(s44_n_articles + 1) / s44_n_articles * 100
for column, color in [("exposures", "#2878A5"), ("clicks", "#C04D33")]:
    counts = s44_exposed_articles[column].sort_values(ascending=False).to_numpy()
    if counts.sum() > 0:
        ax.plot(s44_rank_pct, np.r_[0, counts.cumsum() / counts.sum() * 100],
                label=column.capitalize(), color=color)
ax.plot([0, 100], [0, 100], "--", color="gray", label="Equal distribution")
ax.set(title="Article popularity concentration",
       xlabel="Cumulative % of exposed articles (ranked separately)",
       ylabel="Cumulative % of exposures or clicks", xlim=(0, 100), ylim=(0, 100))
ax.legend()
ax.grid(alpha=0.2)
s44_fig_popularity.tight_layout()
plt.show()
print("Curves rising more sharply above the diagonal indicate greater concentration.")
print("Click counts also reflect exposure opportunities; they are not pure preference measures.")


### 4.4.3 Time gaps between consecutive logged impressions

Sort impressions by user and timestamp, then subtract each user's preceding impression time. Exclude each user's first impression because it has no preceding event in this file. Users with one impression contribute no gaps.

Keep zero gaps for same-timestamp events in summary statistics. The logarithmic histogram shows only positive gaps and states how many zeros it omits. These gaps do not measure intervals between historical clicks, reading duration, or events outside this observation window.


In [ ]:
s44_ordered = s44.sort_values(["user_id", "time_parsed"], kind="stable")
s44_gap_hours = (
    s44_ordered.groupby("user_id", sort=False)["time_parsed"]
    .diff().dt.total_seconds().div(3600).dropna()
)
assert len(s44_gap_hours) == len(s44) - s44["user_id"].nunique()
assert s44_gap_hours.ge(0).all()
s44_zero_gaps = int(s44_gap_hours.eq(0).sum())
s44_positive_gaps = s44_gap_hours.loc[s44_gap_hours > 0]
s44_gap_summary = s44_gap_hours.describe(
    percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]
).to_frame("Gap between consecutive impressions (hours)")
display(s44_gap_summary.round(4))
print(f"Users with at least two logged impressions: {s44_impressions_per_user.ge(2).sum():,}")
print(f"Users with only one logged impression: {s44_impressions_per_user.eq(1).sum():,}")
print(f"Eligible within-user gaps: {len(s44_gap_hours):,}")
if len(s44_gap_hours):
    print(f"Zero-hour gaps (included in statistics): {s44_zero_gaps:,} "
          f"({100 * s44_zero_gaps / len(s44_gap_hours):.2f}%)")

s44_fig_gaps, ax = plt.subplots(figsize=(8, 4))
if len(s44_positive_gaps):
    lo, hi = s44_positive_gaps.min(), s44_positive_gaps.max()
    if lo == hi:
        lo, hi = lo / 2, hi * 2
    ax.hist(s44_positive_gaps, bins=np.geomspace(lo, hi, 41),
            color="#2878A5", edgecolor="white")
    ax.set_xscale("log")
    ax.set_xlabel("Hours between consecutive impressions (log scale)")
    ax.set_ylabel("Number of positive gaps")
else:
    ax.text(0.5, 0.5, "No positive within-user gaps in this data.",
            ha="center", va="center", transform=ax.transAxes)
ax.set_title(f"Within-user impression gaps — {s44_zero_gaps:,} zero gaps omitted from plot")
s44_fig_gaps.tight_layout()
plt.show()
